In [2]:
from faker import Faker
import pandas as pd
import numpy as np
import kagglehub
import os
import random
import pyarrow as pa 
import pyarrow.parquet as pq 


In [5]:
fake = Faker('pl_PL') # Polsa lokalizacja

path = kagglehub.dataset_download("vipullrathod/fish-market")
print("Path to dataset files:", path)
print(os.listdir(path))

csv_path = os.path.join(path, "Fish.csv")
df = pd.read_csv(csv_path)

dfBream = df[df['Species'] == 'Bream']
dfBream_mW, dfBream_stdW = dfBream['Weight'].mean(), dfBream['Weight'].std()
dfBream_mH, dfBream_stdH = dfBream['Height'].mean(), dfBream['Height'].std()

def generate_bream_data(n_samples=50):
    weights = np.random.normal(dfBream_mW, dfBream_stdW, n_samples)
    weights = np.clip(weights, 242, 1100)

    data = []

    for i in range(n_samples):
        if random.random() < 0.05:
            id_ryby = ""
        else:
            id_ryby = f"BRM-{fake.unique.random_int(1000, 9999)}"
        record = {
            "ID_Ryby": id_ryby,
            "Data_Polowu": fake.date_between(start_date='-1y', end_date='today'),
            "Rybak": fake.name(),
            "Lokalizacja": fake.city(),
            "Waga_g": round(weights[i], 2),
            "Stan_Zdrowia": fake.random_element(elements=('Zdrowa', 'Zdrowa', 'Zdrowa', 'Chora')),
            "Komentarz": fake.sentence(nb_words=5)
        }
        data.append(record)

    return pd.DataFrame(data)

dfBream = generate_bream_data(100)
print(dfBream)

Path to dataset files: C:\Users\janva\.cache\kagglehub\datasets\vipullrathod\fish-market\versions\1
['Fish.csv']
     ID_Ryby Data_Polowu                    Rybak           Lokalizacja  \
0   BRM-7158  2025-11-27  Maksymilian Bączkiewicz               Rzeszów   
1             2025-06-29              Klara Ficoń               Jarocin   
2   BRM-4033  2025-10-25          Aniela Opyrchał      Wodzisław Śląski   
3   BRM-9298  2025-10-07         Anastazja Janiga                Krosno   
4   BRM-4947  2025-09-15                Ida Buśko          Stalowa Wola   
..       ...         ...                      ...                   ...   
95  BRM-7050  2025-10-27            Sylwia Marosz               Wrocław   
96  BRM-1389  2026-02-18            Maciej Trybuś             Lubliniec   
97  BRM-4424  2025-12-06          Monika Pacholak                 Ząbki   
98  BRM-7708  2025-06-10         Adam Gabryelczyk               Siedlce   
99  BRM-5302  2025-08-13        pan Cezary Aleksa  Nowy Dwór M

In [ ]:
dfBream['Data_Polowa'] = pd.to_datetime(dfBream['Data_Polowu'])
dfBream['Rok'] = dfBream['Data_Polowa'].dt.year
dfBream['Miesiac'] = dfBream['Data_Polowa'].dt.month

# konwersja do PyArrow
table = pa.Table.from_pandas(dfBream)

In [ ]:
pq.write_to_dataset(
    table,
    root_path = "./lakehouse/bream_data/",
    partition_cols=['Rok', 'Miesiac'],
    existing_data_behavior='overwrite_or_ignore'
)

print("Dane zapisane pomyślnie")